<a href="https://colab.research.google.com/github/w3aarush/DR_Classification_NIT_MCA_Project/blob/main/Version_2_EffnetSVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing Libraries

In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import time
import cv2
import os

from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2S, preprocess_input
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, accuracy_score
from google.colab.patches import cv2_imshow
# from cuml import SVC # for python 3.11

## Dataset Loading

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

In [5]:
# from drive
base_dir = '/content/drive/MyDrive/aptos2019/'
train_img = '/content/drive/MyDrive/aptos2019/train_images/train_images/'
validation_img = '/content/drive/MyDrive/aptos2019/val_images/val_images/'
test_img = '/content/drive/MyDrive/aptos2019/test_images/test_images/'

train_csv = '/content/drive/MyDrive/aptos2019/train_1.csv'
test_csv = '/content/drive/MyDrive/aptos2019/test.csv'
valid_csv = '/content/drive/MyDrive/aptos2019/valid.csv'

In [6]:
def load_data():
    # train = pd.read_csv('/content/aptos2019/train_1.csv', encoding='utf-8')
    # test = pd.read_csv('/content/aptos2019/test.csv', encoding='utf-8')
    # valid = pd.read_csv('/content/aptos2019/valid.csv')

    train = pd.read_csv(train_csv, encoding='utf-8') # from drive
    test = pd.read_csv(test_csv, encoding='utf-8') # from drive
    valid = pd.read_csv(valid_csv) # from drive

    # train_dir = '/content/aptos2019/train_images/train_images/'
    # test_dir = '/content/aptos2019/test_images/test_images/'
    # valid_dir = '/content/aptos2019/val_images/val_images/'

    train_dir = train_img # from drive
    test_dir = test_img # from drive
    valid_dir = validation_img # from drive

    # construct file paths directly within function:
    train['image_path'] = train_dir + train['id_code'] + '.png'
    test['image_path'] = test_dir + test['id_code'] + '.png'
    valid['image_path'] = valid_dir + valid['id_code'] + '.png'

    train['train_images'] = train['id_code'] + '.png'
    test['test_images'] = test['id_code'] + '.png'
    valid['valid_images'] = valid['id_code'] + '.png'

    train['diagnosis'] = train['diagnosis'].astype(str)
    # train['target'] = [1 if x >= 1 else 0 for x in train['diagnosis']]
    # train['target'] = train['target'].astype(str)
    test['diagnosis'] = test['diagnosis'].astype(str)
    # test['target'] = [1 if x >= 1 else 0 for x in test['diagnosis']]
    # test['target'] = test['target'].astype(str)
    valid['diagnosis'] = valid['diagnosis'].astype(str)
    # valid['target'] = [1 if x >= 1 else 0 for x in valid['diagnosis']]
    # valid['target'] = valid['target'].astype(str)

    return train, test, valid

In [7]:
train_df, test_df, valid_df = load_data()

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_code       2930 non-null   object
 1   diagnosis     2930 non-null   object
 2   image_path    2930 non-null   object
 3   train_images  2930 non-null   object
dtypes: object(4)
memory usage: 91.7+ KB


In [ ]:
train_df['diagnosis'].value_counts()

,count
diagnosis,
0,1434
2,808
1,300
4,234
3,154


In [ ]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id_code      366 non-null    object
 1   diagnosis    366 non-null    object
 2   image_path   366 non-null    object
 3   test_images  366 non-null    object
dtypes: object(4)
memory usage: 11.6+ KB


In [ ]:
test_df['diagnosis'].value_counts()

,count
diagnosis,
0,199
2,87
4,33
1,30
3,17


In [ ]:
valid_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_code       366 non-null    object
 1   diagnosis     366 non-null    object
 2   image_path    366 non-null    object
 3   valid_images  366 non-null    object
dtypes: object(4)
memory usage: 11.6+ KB


In [ ]:
valid_df['diagnosis'].value_counts()

,count
diagnosis,
0,172
2,104
1,40
4,28
3,22


In [9]:
NUM_CLASSES = 5
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
epochs = 20

## Image Preprocessing

### CLAHE

In [11]:
def apply_clahe_to_image_tensor(image_tensor):
    # image_tensor has shape (H, W, C) for a single image, float32, in [0, 255]
    # Convert to uint8 for OpenCV operations
    image_np_uint8 = tf.cast(image_tensor, tf.uint8).numpy()

    # Convert RGB to LAB color space to apply CLAHE only to the Lightness channel
    lab = cv2.cvtColor(image_np_uint8, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    # Apply CLAHE to the L channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)) # You can adjust clipLimit and tileGridSize
    cl = clahe.apply(l_channel)

    # Merge the CLAHE enhanced L channel with the original A and B channels
    limg = cv2.merge((cl, a_channel, b_channel))

    # Convert LAB back to RGB
    final_img_np_rgb = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

    # Convert back to float32 and return as TensorFlow tensor
    return tf.convert_to_tensor(final_img_np_rgb, dtype=tf.float32)

# Wrapper function to apply CLAHE to a batch of images using tf.map_fn
def apply_clahe_batch(images_batch_tensor):
    # tf.map_fn applies a function to each element (image) in the batch
    return tf.map_fn(
        fn=lambda img: tf.py_function(apply_clahe_to_image_tensor, [img], tf.float32),
        elems=images_batch_tensor,
        fn_output_signature=tf.TensorSpec(images_batch_tensor.shape[1:], dtype=tf.float32)
    )

### Augmentation Pipeline

In [12]:
img_augmentation = Sequential(
    [
        # tf.keras.layers.Lambda(apply_clahe_batch), # Apply CLAHE first
        tf.keras.layers.RandomRotation(factor=(-0.15, 0.15)),
        tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
        tf.keras.layers.RandomFlip(),
        tf.keras.layers.RandomContrast(factor=0.6),
    ],
    name="img_augmentation",
)

### Verifying `img_augmentation`

In [ ]:
# Select a random image from the training DataFrame
sample_image_path = train_df.sample(1).iloc[0]['image_path']

# Load the image using OpenCV (or PIL/Keras load_img)
original_img = cv2.imread(sample_image_path)
original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB
original_img = cv2.resize(original_img, IMG_SIZE) # Resize to target size

# Convert to TensorFlow tensor and add batch dimension
img_tensor = tf.convert_to_tensor(original_img, dtype=tf.float32)
img_tensor = tf.expand_dims(img_tensor, axis=0) # Add batch dimension

# Apply the augmentation sequence
augmented_img_tensor = img_augmentation(img_tensor, training=True) # training=True enables random augmentations

# Convert augmented image back to NumPy array for display
augmented_img = augmented_img_tensor.numpy()[0]

# Ensure values are in displayable range (e.g., [0, 255] for uint8 or [0, 1] for float)
# The img_augmentation layers (like RandomContrast) keep values generally within 0-255.
# If CLAHE or other ops push values outside, it's good practice to clip or normalize for display.
augmented_img = np.clip(augmented_img, 0, 255).astype(np.uint8)

# Plot original and augmented images
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(original_img)
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(augmented_img)
plt.title('Augmented Image')
plt.axis('off')

plt.show()

In [ ]:
import os
import shutil
from tqdm import tqdm

# Ensure configuration variables are set
NUM_CLASSES = 5
organized_train_dir = '/content/drive/MyDrive/aptos2019_organised'

# Create subdirectories for each class (0 to 4)
for i in range(NUM_CLASSES):
    class_path = os.path.join(organized_train_dir, str(i))
    os.makedirs(class_path, exist_ok=True)

print(f"Organizing images into {organized_train_dir}...")

# Iterate through the training dataframe and copy images
for index, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    src_path = row['image_path']
    class_label = str(row['diagnosis'])
    filename = os.path.basename(src_path)
    dst_path = os.path.join(organized_train_dir, class_label, filename)

    # Copy the file if it exists
    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)
    else:
        print(f"Warning: {src_path} not found.")

print("\nImage organization complete.")

Organizing images into /content/drive/MyDrive/aptos2019_organised...


100%|██████████| 2930/2930 [18:12<00:00,  2.68it/s]


Image organization complete.


In [ ]:
train_0 = train_df[train_df['diagnosis']== '0'][['id_code','diagnosis']]
train_1 = train_df[train_df['diagnosis']== '1'][['id_code','diagnosis']]
train_2 = train_df[train_df['diagnosis']== '2'][['id_code','diagnosis']]
train_3 = train_df[train_df['diagnosis']== '3'][['id_code','diagnosis']]
train_4 = train_df[train_df['diagnosis']== '4'][['id_code','diagnosis']]

In [ ]:
train_0.to_csv('/content/drive/MyDrive/aptos2019_organised/train/train_0.csv', index=False)
train_1.to_csv('/content/drive/MyDrive/aptos2019_organised/train/train_1.csv', index=False)
train_2.to_csv('/content/drive/MyDrive/aptos2019_organised/train/train_2.csv', index=False)
train_3.to_csv('/content/drive/MyDrive/aptos2019_organised/train/train_3.csv', index=False)
train_4.to_csv('/content/drive/MyDrive/aptos2019_organised/train/train_4.csv', index=False)

In [ ]:
val_0 = valid_df[valid_df['diagnosis'] == '0'][['id_code','diagnosis']]
val_1 = valid_df[valid_df['diagnosis'] == '1'][['id_code','diagnosis']]
val_2 = valid_df[valid_df['diagnosis'] == '2'][['id_code','diagnosis']]
val_3 = valid_df[valid_df['diagnosis'] == '3'][['id_code','diagnosis']]
val_4 = valid_df[valid_df['diagnosis'] == '4'][['id_code','diagnosis']]

In [ ]:
val_0.to_csv('/content/drive/MyDrive/aptos2019_organised/val/val_0.csv', index=False)
val_1.to_csv('/content/drive/MyDrive/aptos2019_organised/val/val_1.csv', index=False)
val_2.to_csv('/content/drive/MyDrive/aptos2019_organised/val/val_2.csv', index=False)
val_3.to_csv('/content/drive/MyDrive/aptos2019_organised/val/val_3.csv', index=False)
val_4.to_csv('/content/drive/MyDrive/aptos2019_organised/val/val_4.csv', index=False)

In [ ]:
test_0 = test_df[test_df['diagnosis'] == '0'][['id_code', 'diagnosis']]
test_1 = test_df[test_df['diagnosis'] == '1'][['id_code', 'diagnosis']]
test_2 = test_df[test_df['diagnosis'] == '2'][['id_code', 'diagnosis']]
test_3 = test_df[test_df['diagnosis'] == '3'][['id_code', 'diagnosis']]
test_4 = test_df[test_df['diagnosis'] == '4'][['id_code', 'diagnosis']]

In [ ]:
test_0.to_csv('/content/drive/MyDrive/aptos2019_organised/test/test_0.csv', index=False)
test_1.to_csv('/content/drive/MyDrive/aptos2019_organised/test/test_1.csv', index=False)
test_2.to_csv('/content/drive/MyDrive/aptos2019_organised/test/test_2.csv', index=False)
test_3.to_csv('/content/drive/MyDrive/aptos2019_organised/test/test_3.csv', index=False)
test_4.to_csv('/content/drive/MyDrive/aptos2019_organised/test/test_4.csv', index=False)

In [1]:
file_path = '/content/drive/MyDrive/NIT_DR_Project/aptos2019_organised'

In [13]:
import os
import shutil
import math
from tqdm import tqdm
import tensorflow as tf
import numpy as np
import cv2
import pandas as pd # Ensure pandas is imported as it's used in the logic

# --- BEGIN ADDED CODE TO ADDRESS NAMEOCCUR ---

# From cell 0LE-ZGs1gotQ: load_data function definition
def load_data():
    # Assuming train_csv, test_csv, valid_csv are defined globally from cell 75PMwjDtDM-g
    train = pd.read_csv(train_csv, encoding='utf-8')
    test = pd.read_csv(test_csv, encoding='utf-8')
    valid = pd.read_csv(valid_csv)

    # Assuming train_img, test_img, validation_img are defined globally from cell 75PMwjDtDM-g
    train_dir = train_img
    test_dir = test_img
    valid_dir = validation_img

    # construct file paths directly within function:
    train['image_path'] = train_dir + train['id_code'] + '.png'
    test['image_path'] = test_dir + test['id_code'] + '.png'
    valid['image_path'] = valid_dir + valid['id_code'] + '.png'

    train['train_images'] = train['id_code'] + '.png'
    test['test_images'] = test['id_code'] + '.png'
    valid['valid_images'] = valid['id_code'] + '.png'

    train['diagnosis'] = train['diagnosis'].astype(str)
    test['diagnosis'] = test['diagnosis'].astype(str)
    valid['diagnosis'] = valid['diagnosis'].astype(str)

    return train, test, valid

# From cell MjxQMylNjg7S: calling load_data to define dataframes
train_df, test_df, valid_df = load_data()

# From cell A0BXofBqK9cx: constants definition
NUM_CLASSES = 5
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
epochs = 20

# --- END ADDED CODE TO ADDRESS NAMEOCCUR ---


# Define the new base directory for the balanced dataset in Google Drive
balanced_base_dir = '/content/drive/MyDrive/balanced_aptos2019'

# Ensure the main balanced directory exists
os.makedirs(balanced_base_dir, exist_ok=True)

# Define splits using the already loaded dataframes
splits = {
    "train": train_df,
    "valid": valid_df,
    "test": test_df
}

# Function to load and preprocess a single image for augmentation
def load_image_for_augmentation(image_path, target_size=IMG_SIZE):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: Could not read image at {image_path}")
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size)
    img_tensor = tf.convert_to_tensor(img, dtype=tf.float32)
    img_tensor = tf.expand_dims(img_tensor, axis=0) # Add batch dimension
    return img_tensor

# Function to balance and save images for a given split
def balance_and_save_split(df, split_name, base_output_dir, num_classes=NUM_CLASSES, img_augmentation_model=None):
    print(f"\n--- Balancing and saving {split_name} split ---")

    # Create split-specific output directory
    split_output_dir = os.path.join(base_output_dir, split_name)
    os.makedirs(split_output_dir, exist_ok=True)

    # Get class counts
    class_counts = df['diagnosis'].value_counts().sort_index()
    print(f"Original {split_name} class counts:\n{class_counts}")

    # Determine the target count (maximum count)
    target_count = class_counts.max()
    print(f"Target count per class for {split_name}: {target_count}")

    # List to store information about new augmented images for creating a new DataFrame later
    augmented_image_data = []

    # Iterate through each class
    for i in range(num_classes):
        class_label = str(i)
        class_output_dir = os.path.join(split_output_dir, class_label)
        os.makedirs(class_output_dir, exist_ok=True)

        class_df = df[df['diagnosis'] == class_label]
        current_class_count = len(class_df)

        print(f"Class {class_label}: Current count = {current_class_count}, Target count = {target_count}")

        # Copy original images and collect their paths for the new dataframe
        for idx, row in tqdm(class_df.iterrows(), total=current_class_count, desc=f"Copying original {split_name} class {class_label} images"):
            src_path = row['image_path']
            filename = os.path.basename(src_path)
            dst_path = os.path.join(class_output_dir, filename)
            if os.path.exists(src_path):
                shutil.copy(src_path, dst_path)
                # Add original image info to augmented_image_data list
                augmented_image_data.append({'id_code': row['id_code'], 'diagnosis': class_label, 'image_path': dst_path})
            else:
                print(f"Warning: Original image {src_path} not found.")

        # Augment images if needed
        needed_augmentations = target_count - current_class_count
        if needed_augmentations > 0 and img_augmentation_model is not None:
            print(f"Augmenting {needed_augmentations} images for {split_name} class {class_label}...")

            # Select images to augment (cycle through existing images)
            images_to_augment_paths = class_df['image_path'].tolist()
            if not images_to_augment_paths:
                print(f"Warning: No images to augment from for {split_name} class {class_label}.")
                continue

            augmentation_idx = 0
            for k in tqdm(range(needed_augmentations), desc=f"Generating augmented {split_name} class {class_label} images"):
                original_img_path = images_to_augment_paths[augmentation_idx % len(images_to_augment_paths)]
                img_tensor = load_image_for_augmentation(original_img_path)

                if img_tensor is not None:
                    # Apply augmentation
                    augmented_img_tensor = img_augmentation_model(img_tensor, training=True)
                    augmented_img = augmented_img_tensor.numpy()[0]

                    # Convert back to uint8 for saving
                    augmented_img = np.clip(augmented_img, 0, 255).astype(np.uint8)

                    # Generate unique filename for augmented image
                    original_filename_base = os.path.splitext(os.path.basename(original_img_path))[0]
                    augmented_filename = f"{original_filename_base}_aug_{k:04d}.png"
                    dst_path = os.path.join(class_output_dir, augmented_filename)

                    # Save the augmented image
                    cv2.imwrite(dst_path, cv2.cvtColor(augmented_img, cv2.COLOR_RGB2BGR))
                    augmented_image_data.append({'id_code': f'{original_filename_base}_aug_{k:04d}', 'diagnosis': class_label, 'image_path': dst_path})
                augmentation_idx += 1
        elif img_augmentation_model is None and needed_augmentations > 0:
            print(f"Warning: img_augmentation_model is None, cannot augment for {split_name} class {class_label}.")

    print(f"{split_name} split balancing complete.")
    # Return a new DataFrame with paths to all images (original + augmented) for this split
    return pd.DataFrame(augmented_image_data)


# Process each split
balanced_dataframes = {}
for split_name, df in splits.items():
    # Ensure 'img_augmentation' is accessible (it's defined in a prior cell)
    if 'img_augmentation' in locals() or 'img_augmentation' in globals():
        balanced_dataframes[split_name] = balance_and_save_split(df, split_name, balanced_base_dir, img_augmentation_model=img_augmentation)
    else:
        print("Error: 'img_augmentation' model not found. Please ensure it's defined in a preceding cell.")
        break

print("\nAll datasets balanced and saved!")

# Optional: Display the new balanced class counts if dataframes were generated successfully
for split_name, balanced_df in balanced_dataframes.items():
    if not balanced_df.empty:
        print(f"\nBalanced {split_name} class counts:\n{balanced_df['diagnosis'].value_counts().sort_index()}")
    else:
        print(f"\nNo balanced dataframe generated for {split_name}.")


--- Balancing and saving train split ---
Original train class counts:
diagnosis
0    1434
1     300
2     808
3     154
4     234
Name: count, dtype: int64
Target count per class for train: 1434
Class 0: Current count = 1434, Target count = 1434


Copying original train class 0 images: 100%|██████████| 1434/1434 [10:23<00:00,  2.30it/s]


Class 1: Current count = 300, Target count = 1434


Copying original train class 1 images: 100%|██████████| 300/300 [02:15<00:00,  2.21it/s]


Augmenting 1134 images for train class 1...


...

Class 4: Current count = 33, Target count = 199


Copying original test class 4 images: 100%|██████████| 33/33 [00:16<00:00,  2.05it/s]


Augmenting 166 images for test class 4...


Generating augmented test class 4 images: 100%|██████████| 166/166 [00:39<00:00,  4.24it/s]

test split balancing complete.

All datasets balanced and saved!

Balanced train class counts:
diagnosis
0    1434
1    1434
2    1434
3    1434
4    1434
Name: count, dtype: int64

Balanced valid class counts:
diagnosis
0    172
1    172
2    172
3    172
4    172
Name: count, dtype: int64

Balanced test class counts:
diagnosis
0    199
1    199
2    199
3    199
4    199
Name: count, dtype: int64


### Saving Balanced DataFrames to CSV

In [14]:
import os

# Define paths for saving the new balanced CSVs
balanced_train_csv = os.path.join(balanced_base_dir, 'balanced_train.csv')
balanced_valid_csv = os.path.join(balanced_base_dir, 'balanced_valid.csv')
balanced_test_csv = os.path.join(balanced_base_dir, 'balanced_test.csv')

# Save each balanced DataFrame to CSV
balanced_dataframes['train'].to_csv(balanced_train_csv, index=False)
balanced_dataframes['valid'].to_csv(balanced_valid_csv, index=False)
balanced_dataframes['test'].to_csv(balanced_test_csv, index=False)

print(f"Balanced train data saved to: {balanced_train_csv}")
print(f"Balanced validation data saved to: {balanced_valid_csv}")
print(f"Balanced test data saved to: {balanced_test_csv}")

Balanced train data saved to: /content/drive/MyDrive/balanced_aptos2019/balanced_train.csv
Balanced validation data saved to: /content/drive/MyDrive/balanced_aptos2019/balanced_valid.csv
Balanced test data saved to: /content/drive/MyDrive/balanced_aptos2019/balanced_test.csv


### Displaying Head of Balanced DataFrames

In [15]:
print("\nBalanced Train DataFrame Head:")
display(balanced_dataframes['train'].head())

print("\nBalanced Validation DataFrame Head:")
display(balanced_dataframes['valid'].head())

print("\nBalanced Test DataFrame Head:")
display(balanced_dataframes['test'].head())


Balanced Train DataFrame Head:


,id_code,diagnosis,image_path
0,1b3647865779,0,/content/drive/MyDrive/balanced_aptos2019/trai...
1,1b398c0494d1,0,/content/drive/MyDrive/balanced_aptos2019/trai...
2,1b862fb6f65d,0,/content/drive/MyDrive/balanced_aptos2019/trai...
3,1b8701231c8f,0,/content/drive/MyDrive/balanced_aptos2019/trai...
4,1c13a1483f4a,0,/content/drive/MyDrive/balanced_aptos2019/trai...



Balanced Validation DataFrame Head:


,id_code,diagnosis,image_path
0,002c21358ce6,0,/content/drive/MyDrive/balanced_aptos2019/vali...
1,005b95c28852,0,/content/drive/MyDrive/balanced_aptos2019/vali...
2,0097f532ac9f,0,/content/drive/MyDrive/balanced_aptos2019/vali...
3,00cc2b75cddd,0,/content/drive/MyDrive/balanced_aptos2019/vali...
4,00f6c1be5a33,0,/content/drive/MyDrive/balanced_aptos2019/vali...



Balanced Test DataFrame Head:


,id_code,diagnosis,image_path
0,e4dcca36ceb4,0,/content/drive/MyDrive/balanced_aptos2019/test...
1,e50b0174690d,0,/content/drive/MyDrive/balanced_aptos2019/test...
2,e5197d77ec68,0,/content/drive/MyDrive/balanced_aptos2019/test...
3,e529c5757d64,0,/content/drive/MyDrive/balanced_aptos2019/test...
4,e582e56e7942,0,/content/drive/MyDrive/balanced_aptos2019/test...


### Setting Up Data Generators for Balanced Dataset

In [17]:
# Define the new image directories for the balanced dataset
balanced_train_img_dir = os.path.join(balanced_base_dir, 'train')
print(balanced_train_img_dir)
balanced_valid_img_dir = os.path.join(balanced_base_dir, 'valid')
print(balanced_valid_img_dir)
balanced_test_img_dir = os.path.join(balanced_base_dir, 'test')
print(balanced_test_img_dir)

# Data Generator for Training (with augmentation)
# The img_augmentation layer handles augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input # Assuming preprocess_input from EfficientNetV2S is intended
)

# Data Generator for Validation and Test (only preprocessing)
val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input # Assuming preprocess_input from EfficientNetV2S is intended
)

# Create flow_from_directory generators
# Use the balanced dataframes as the source
train_generator = train_datagen.flow_from_directory(
    balanced_train_img_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True # Shuffle training data
)

validation_generator = val_test_datagen.flow_from_directory(
    balanced_valid_img_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # Do not shuffle validation data
)

test_generator = val_test_datagen.flow_from_directory(
    balanced_test_img_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # Do not shuffle test data
)

print("\n--- Data Generator Summary ---")
print("Train Generator Class Indices:", train_generator.class_indices)
print("Validation Generator Class Indices:", validation_generator.class_indices)
print("Test Generator Class Indices:", test_generator.class_indices)

print(f"\nFound {train_generator.samples} training images belonging to {train_generator.num_classes} classes.")
print(f"Found {validation_generator.samples} validation images belonging to {validation_generator.num_classes} classes.")
print(f"Found {test_generator.samples} test images belonging to {test_generator.num_classes} classes.")

/content/drive/MyDrive/balanced_aptos2019/train
/content/drive/MyDrive/balanced_aptos2019/valid
/content/drive/MyDrive/balanced_aptos2019/test
Found 7170 images belonging to 5 classes.
Found 860 images belonging to 5 classes.
Found 995 images belonging to 5 classes.

--- Data Generator Summary ---
Train Generator Class Indices: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}
Validation Generator Class Indices: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}
Test Generator Class Indices: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}

Found 7170 training images belonging to 5 classes.
Found 860 validation images belonging to 5 classes.
Found 995 test images belonging to 5 classes.


###################################################################

In [30]:
df = pd.read_csv('/content/drive/MyDrive/balanced_aptos2019/balanced_train.csv')

In [34]:
image_path2 = '/content/drive/MyDrive/balanced_aptos2019/train/' + df['diagnosis'].astype(str)

In [37]:
df['path2'] = image_path2

In [46]:
base_dir = '/content/drive/MyDrive/NIT_DR_Project/balanced_aptos2019/'
train_image = '/content/drive/MyDrive/NIT_DR_Project/balanced_aptos2019/train/'
validation_image = '/content/drive/MyDrive/NIT_DR_Project/balanced_aptos2019/valid/'
test_image = '/content/drive/MyDrive/NIT_DR_Project/balanced_aptos2019/test/'

train_csv = '/content/drive/MyDrive/NIT_DR_Project/balanced_aptos2019/balanced_train.csv'
test_csv = '/content/drive/MyDrive/NIT_DR_Project/balanced_aptos2019/balanced_test.csv'
valid_csv = '/content/drive/MyDrive/NIT_DR_Project/balanced_aptos2019/balanced_valid.csv'

def load_data():
    train = pd.read_csv(train_csv, encoding='utf-8')
    valid = pd.read_csv(valid_csv, encoding='utf-8')
    test = pd.read_csv(test_csv, encoding='utf-8')

    # Convert 'diagnosis' to string type BEFORE path construction
    train['diagnosis'] = train['diagnosis'].astype(str)
    valid['diagnosis'] = valid['diagnosis'].astype(str)
    test['diagnosis'] = test['diagnosis'].astype(str)

    train_dir = train_image
    valid_dir = validation_image
    test_dir = test_image

    # Construct file paths using the now-string 'diagnosis' column
    train['image_path'] = train_image + train['diagnosis'] + '/' + train['id_code'] + '.png'
    valid['image_path'] = validation_image + valid['diagnosis'] + '/' + valid['id_code'] + '.png'
    test['image_path'] = test_image + test['diagnosis'] + '/' + test['id_code'] + '.png'

    train['train_images'] = train['id_code']+'.png'
    valid['valid_images']  = valid['id_code']+'.png'
    test['test_images'] = test['id_code']+'.png'

    return train, valid, test


In [47]:
train_df, valid_df, test_df = load_data()

In [50]:
print(train_df['diagnosis'].value_counts())
print(valid_df['diagnosis'].value_counts())
print(test_df['diagnosis'].value_counts())

diagnosis
0    1434
1    1434
2    1434
3    1434
4    1434
Name: count, dtype: int64
diagnosis
0    172
1    172
2    172
3    172
4    172
Name: count, dtype: int64
diagnosis
0    199
1    199
2    199
3    199
4    199
Name: count, dtype: int64
